# Reproduce — González-Palacio et al. (2023), IEEE IoT-J

**Run All** does the whole project: the pipeline (~10 min), the tests, then every table and figure. Inputs: `data/LoRaWAN_PathLossMeasurements.csv` and `paper_digitized/` (the paper's curves, for the hollow markers). Outputs land in `tables/`, `figures/`, `models/`. To only redraw from existing outputs, skip section 1.

## 1. Run the pipeline

data → conventional models → CPLS models (MLR, ANN, SVR, RF) → residuals → ADR and energy → headline claims. Every choice the paper leaves open is in `run_final.CONFIG`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
import run_final
print({k: run_final.CONFIG[k] for k in ('algorithm1_variant', 'delivery_rule', 'sf_max', 'restore_frac_rssi_only', 'restore_frac_coupled')})
run_final.main()

## 2. Tests

Algorithm 1 vectorised == scalar transcription (5 tests); CONFIG forwarding, cache invalidation, delivery-rule dependence (3).

In [ ]:
import subprocess
for t in ('tests/test_adr_equivalence.py', 'tests/test_pipeline_behaviour.py'):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True)
    print(t, '->', 'OK' if r.returncode == 0 else 'FAILED'); print(r.stdout.strip()[-600:] if r.returncode == 0 else r.stderr[-1500:])

## 3. Tables and figures

Everything below is read from `tables/*.csv`; figures are drawn by `src/plots.py`.

In [ ]:
import json
import pandas as pd, matplotlib.pyplot as plt
import plots
F = pathlib.Path.cwd() / 'tables'
pd.set_option('display.width', 200, 'display.max_columns', 30, 'display.precision', 3)

## Headline claims — what each verdict rests on

In [ ]:
pd.read_csv(F / 'headline_claims.csv')

## Fig. 4 — conventional models

In [ ]:
plots.fig04(F); plt.show()

In [ ]:
pd.read_csv(F / 'table_iii_conventional.csv')

## Table IV — CPLS models

In [ ]:
pd.read_csv(F / 'table_iv_cpls.csv')

## Fig. 11 — PDR vs link margin

In [ ]:
plots.fig11(F); plt.show()

In [ ]:
pd.read_csv(F / 'fig11_link_margins.csv').pivot(index='scheme', columns='pdr', values='LM_dB')

## Figs. 12–13 — at the paper's operating LMs (figure reconstruction; achieved PDRs differ from the labels)

In [ ]:
plots.fig12(F); plots.fig13(F); plt.show()

In [ ]:
e = pd.read_csv(F / 'fig12_13_energy_toa.csv'); e[e.pdr == 99][['scheme', 'LM_dB', 'pdr_at_LM', 'energy_improvement_pct', 'toa_improvement_pct']]

## Figs. 14–15 — MLR residual QQ plots

In [ ]:
plots.fig14_15(F); plt.show()

In [ ]:
pd.read_csv(F / 'appendix_residual_tests.csv')